# Introduction:

*Stuff to add*

- Contents structure
- Explain what the project is about
- How we carried out the analysis
- What are the tools used



## Data Acquisition:

### Primary: Guardian API
We will collect articles from the Guardian API using commodity-related search terms (e.g. oil, natural gas, copper, wheat, OPEC, energy crisis) filtered by relevant sections (e.g. Business, Environment, World News). For each article, we may extract the publication date, headline, section, tags, word count, and body text. We plan to collect data spanning approximately 2014 - 2026 to capture multiple geopolitical cycles.

### Secondary: Commodity Price Data
To contextualise media coverage against market movements, we will source historical commodity price data from Yahoo Finance (via the yfinance Python library) or publicly available datasets. This will include daily/weekly prices for key commodities such as Brent Crude, natural gas, gold, copper, and wheat.

- Query the Guardian API with commodity-related keywords and display results across the full date range

- Download commodity price time series from Yahoo Finance using yfinance

- Store raw data in structured formats (JSON/CSV) for reproducibility



In [1]:
import json
import requests
import pandas as pd
import time
from datetime import date

with open('keys.json') as f:
    key = json.load(f)

API_KEY = key['guardian']['api_key']
BASE_URL = 'https://content.guardianapis.com'


date_ranges = [
    ("2020-01-01", "2020-06-30"),
    ("2020-07-01", "2020-12-31"),
    ("2021-01-01", "2021-06-30"),
    ("2021-07-01", "2021-12-31"),
    ("2022-01-01", "2022-06-30"),
    ("2022-07-01", "2022-12-31"),
    ("2023-01-01", "2023-06-30"),
    ("2023-07-01", "2023-12-31"),
    ("2024-01-01", "2024-06-30"),
    ("2024-07-01", "2024-12-31"),
    ("2025-01-01", "2025-12-31"),
]

all_results = []

for from_date, to_date in date_ranges:
    for page in range(1, 21):
        parameters = {
            "api-key": API_KEY,
            "q": "oil OR natural gas OR copper OR wheat OR OPEC OR energy crisis",
            "page-size": 50,
            "page": page,
            "show-fields": "bodyText",
            "from-date": from_date,
            "to-date": to_date,
            "order-by": "oldest"
        }

        response = requests.get(f"{BASE_URL}/search", params=parameters)
        response.raise_for_status()
        data = response.json()['response']


        results = data['results']
        total_pages = data['pages']

        for article in results:
            all_results.append({
                'date': article.get('webPublicationDate'),
                'section': article.get('sectionName'),
                'title': article.get('webTitle'),
                'body': article.get('fields', {}).get('bodyText', '')
            })

        if page >= total_pages:
            break

        time.sleep(0.1)

    print(f"  Total so far: {len(all_results)} articles")

df = pd.DataFrame(all_results)
df['date'] = pd.to_datetime(df['date'])
df = df.drop_duplicates(subset=['title'])
df.to_csv('data/guardian_commodities.csv', index=False)
print(f"\nFinal: {len(df)} articles saved")



/Users/pramod/Documents/GitHub/st115/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


  Total so far: 3000 articles
  Total so far: 5171 articles
  Total so far: 7295 articles
  Total so far: 9967 articles
  Total so far: 12967 articles
  Total so far: 15967 articles
  Total so far: 18836 articles
  Total so far: 21744 articles
  Total so far: 24441 articles
  Total so far: 27070 articles
  Total so far: 30070 articles

Final: 30007 articles saved


In [8]:
import yfinance as yf


# Define commodity tickers
# Yahoo Finance uses these symbols for common commodities:
commodities = {
    "Gold":        "GC=F",   # Gold Futures
    "Silver":      "SI=F",   # Silver Futures
    "Crude Oil":   "CL=F",   # WTI Crude Oil Futures
    "Brent Oil":   "BZ=F",   # Brent Crude Futures
    "Natural Gas": "NG=F",   # Natural Gas Futures
    "Copper":      "HG=F",   # Copper Futures
    "Wheat":       "ZW=F",   # Wheat Futures
}

# Download historical data
data = yf.download(
    tickers=list(commodities.values()),
    start="2014-01-01",
    end="2026-03-31",
    interval="1d"       # 1d; daily
)

# Extract closing prices only
close_prices = data["Close"]
close_prices.columns = list(commodities.keys())

df_commodities_prices = pd.DataFrame(close_prices)


print(close_prices.head())


[*********************100%***********************]  7 of 7 completed

                  Gold     Silver    Crude Oil  Brent Oil  Natural Gas  \
Date                                                                     
2014-01-02  107.779999  95.440002  1225.000000     3.4315        4.321   
2014-01-03  106.889999  93.959999  1238.400024     3.4060        4.304   
2014-01-06  106.730003  93.430000  1237.800049     3.4120        4.306   
2014-01-07  107.349998  93.669998  1229.400024     3.4110        4.299   
2014-01-08  107.150002  92.330002  1225.300049     3.3935        4.216   

               Copper   Wheat  
Date                           
2014-01-02  20.098000  597.00  
2014-01-03  20.181999  605.75  
2014-01-06  20.077000  605.75  
2014-01-07  19.764999  602.50  
2014-01-08  19.518000  588.75  
